#### **OPTIMALIDAD DE LA SOLUCION**

In [7]:
import pandas as pd 
import matplotlib.pyplot as plt
import math
from importlib import reload

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")
factibilidad = pd.read_csv("4r.factibilidad.csv")

Empecemos nuevamente guardando los productos y tipos de cajas en listas.

In [8]:
caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")

cajas = [
    Caja(
        caja_id = row["caja_tipo_id"],
        dim_interior_ancho = row["caja_interior_ancho"],
        dim_interior_largo = row["caja_interior_largo"],
        dim_interior_alto = row["caja_interior_alto"],
        costo_unitario = row['costo_unitario_base']
    )
    for _, row in caja_compras_merge.iterrows()
]

cajas[:5]

[<Caja 016d196c89dcfcb306b853a776a879b9 | Int: 248.0 x 383.0 x 224.0mm | Compra Total: 0>,
 <Caja 01a2a319402ed2155292c04d8748e16f | Int: 282.0 x 380.0 x 185.0mm | Compra Total: 0>,
 <Caja 026560e43f3fc6afe0ce89d7ddf61626 | Int: 290.0 x 390.0 x 184.0mm | Compra Total: 0>,
 <Caja 02d7c6680102bd11e067c00c9b71ab9c | Int: 248.0 x 383.0 x 268.0mm | Compra Total: 0>,
 <Caja 0378f85c226113f4ac40fd360217bb8a | Int: 289.0 x 390.0 x 224.0mm | Compra Total: 0>]

In [9]:
operaciones_planta_aux = operaciones_planta.drop('codigo_producto', axis=1) 
prod_op_merge = pd.concat([catalogo_productos, operaciones_planta_aux], axis=1)

productos = [
    Producto(
        codigo_producto = row['codigo_producto'],
        cantidad_paquetes = row['cantidad_paquetes'],
        peso_paquete = row['peso_neto_paquete'],
        demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
        demanda_curitiba = row['volumen_producto_planta_curitiba'],
        demanda_santiago = row['volumen_producto_planta_santiago'],
        demanda_monterrey = row['volumen_producto_planta_monterrey'],
        demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
        dim_producto_ancho = row['dim_producto_ancho'], 
        dim_producto_largo = row['dim_producto_largo'],
        dim_producto_alto = row['dim_producto_alto']
    )
    for _, row in prod_op_merge.iterrows()
]

productos[:5]

[<Producto BR0001 | Dim Prod: 285.0 x 386.0 x 303.0mm | Demanda Total: 1546613>,
 <Producto BR0002 | Dim Prod: 290.0 x 390.0 x 260.0mm | Demanda Total: 139211>,
 <Producto BR0003 | Dim Prod: 287.0 x 388.0 x 164.0mm | Demanda Total: 172506>,
 <Producto BR0004 | Dim Prod: 290.0 x 390.0 x 224.0mm | Demanda Total: 271715>,
 <Producto BR0005 | Dim Prod: 285.0 x 386.0 x 253.0mm | Demanda Total: 7586>]

Agregamos también las cajas asignables por producto en la lista de productos.

In [10]:
# Crear un diccionario de cajas por producto desde el CSV de factibilidad
cajas_por_producto = {}
for _, row in factibilidad.iterrows():
    codigo = row['codigo_producto']
    cajas_str = row['cajas_asignables_id']
    
    if isinstance(cajas_str, str) and cajas_str:
        # Separar por '; ' y limpiar
        cajas_list = [c.strip() for c in cajas_str.split(';') if c.strip()]
        cajas_por_producto[codigo] = cajas_list

# Recorrer la lista de productos en orden y agregar las cajas
for producto in productos:
    if producto.codigo_producto in cajas_por_producto:
        for caja_id in cajas_por_producto[producto.codigo_producto]:
            producto.agregar_caja_asignable(caja_id)

#### **Optimalidad 1: Costos base sin descuentos**

Inicialmente, plantearemos una solución únicamente comparando los costos unitarios base, sin considerar todavía los descuentos por volúmenes anuales.

Como no hay ninguna restricción sobre la cantidad de cajas que podemos comprar de cada tipo, el problema se reduce en encontrar para cada producto el tipo de caja que más le convenga, es decir, el que ofrezca el menor costo.

Empecemos eligiendo un grosor de 4.5mm para todos los tipos de cajas, de manera que la restricción de headspace no supone ningún problema, pues las dimensiones internas de las cajas se diferencian hasta un 10% de las originales.

In [11]:
for caja in cajas:
    caja.elegir_grosor(4.5)

In [12]:
def buscar_caja_por_id(id):
    caja_a_buscar = None
    for caja in cajas:
        if caja.caja_id == id:
            caja_a_buscar = caja
    return caja_a_buscar

def cant_pallets_requeridas(producto, caja):
    cantidad = 0
    plantas = ['buenos_aires', 'curitiba', 'santiago', 'monterrey', 'bakersfield']
    for planta in plantas:
        volumen = getattr(producto, f'demanda_{planta}')
        cajas_por_pallet = caja.cantidad_cajas_por_pallet()
        cantidad += math.ceil(volumen / cajas_por_pallet) # Redondear siempre para arriba
    return cantidad

resultados_sin_descuentos = []

for producto in productos:
    for caja_id in producto.cajas_asignables:
        # Buscamos la caja por su id
        caja = buscar_caja_por_id(caja_id)
        costo_packaging = caja.costo_producto_total(producto)
        cant_pallets = cant_pallets_requeridas(producto, caja)
        costo_flete = 150 * cant_pallets
        costo_total = costo_packaging + costo_flete
        
        resultados_sin_descuentos.append({
            'codigo_producto': producto.codigo_producto,
            'demanda_total': producto.demanda_total(),
            'caja_id': caja_id,
            'utilizacion_pallet': caja.utilizacion_pallet(),
            'utilizacion_caja': caja.utilizacion_caja(producto),
            'costo_packaging': costo_packaging,
            'cant_pallets': cant_pallets,
            'costo_flete': costo_flete,
            'costo_total': costo_total
        })

df_sin_descuentos = pd.DataFrame(resultados_sin_descuentos)
df_sin_descuentos

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,46538be6ce954d7b010ddb9c9fb00f15,0.879040,0.952830,1082629.10,38667,5800050,6882679.10
1,BR0001,1546613,58c5c51d6e39e80f84be2a39cc4677a8,0.838717,1.000000,1082629.10,38667,5800050,6882679.10
2,BR0001,1546613,d0823ee866ead1526b62ea77fb86dde9,0.891516,0.939141,1005298.45,38667,5800050,6805348.45
3,BR0001,1546613,e6369fff40e443b558bbf80d6a5c43f6,0.869903,0.963143,1005298.45,38667,5800050,6805348.45
4,BR0002,139211,057b7f8c3772129fb0b8cccf3c827c80,0.977605,0.909091,90487.15,2901,435150,525637.15
...,...,...,...,...,...,...,...,...,...
3674,BR0421,145803,908589535d3feeff9a55c9b5050b8cec,0.815866,0.973913,94771.95,2604,390600,485371.95
3675,BR0421,145803,c8c092bbf8da5f80452c66c38d27305f,0.795384,1.000000,94771.95,2604,390600,485371.95
3676,BR0421,145803,d20c085d39ceb93f1f9dbe59c29a4786,0.850003,0.933333,94771.95,2604,390600,485371.95
3677,BR0421,145803,ef0525f0911fd36073c2b10f4d81dfd5,0.832935,0.953191,94771.95,2604,390600,485371.95


#### **Optimalidad 2: Costos con descuentos**